# Concept Tasks Pipeline

This notebook processes all questions from a json file, generates answers using a specified model, judges the answers using Gemini, and calculates accuracy metrics.

## Pipeline Steps:
1. **Load Concept Tasks**
2. **Answer Questions** 
3. **Judge Answers** 
4. **Calculate Metrics**



In [1]:
# from utils import define_task, define_incorrect, classify_task, generate_task, edit_task, extract_verdict_counts, extract_verdict_counts_classify, get_concept_accuracies,  all_concepts
# from utils import judge_definitions, judge_generate, judge_edits
import os
import json

In [2]:
MODEL_CHOICES = [
    "meta-llama/Llama-3.2-3B-Instruct", 
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct"
]
model = MODEL_CHOICES[2] 

In [3]:
print(f"Using model: {model}")

Using model: Qwen/Qwen2.5-7B-Instruct


In [4]:
# Load the concept tasks JSON file
with open('concept_tasks_json.json', 'r', encoding='utf-8') as f:
    concept_tasks = json.load(f)

print("Loaded concept tasks for concepts:", list(concept_tasks['concepts'].keys()))

Loaded concept tasks for concepts: ['analogy', 'haiku', 'paradox', 'spotlight_effect', 'black_and_white_thinking', 'catastrophizing', 'endowment_effect', 'fundamental_attribution_error', 'sunk_cost_fallacy', 'pareto_optimality']


## Pipeline: Answer All Questions from JSON

In [5]:
def answer_all_questions(concept_tasks, model):
    from prompts import GENERAL_PROMPT
    from utils import generate_inference
    
    results = {}
    
    for concept_name, concept_data in concept_tasks['concepts'].items():
        print(f"\n{'='*60}")
        print(f"Processing concept: {concept_name}")
        print(f"{'='*60}")
        
        results[concept_name] = {}
        
        for level_name, level_data in concept_data.items():
            print(f"\n  Level: {level_name}")
            results[concept_name][level_name] = {}
            
            if 'questions' in level_data:
                questions = level_data['questions']
                for i, question in enumerate(questions, 1):
                    if isinstance(question, dict):
                        situation = question.get('situation', '')
                        q_text = question.get('question', '')
                        full_question = f"{situation}\n\n{q_text}" if situation else q_text
                        prompt = GENERAL_PROMPT + f"\n\n{full_question}"
                        answer = generate_inference(prompt, model)
                        results[concept_name][level_name][f"Q{i}"] = {
                            "question": full_question,
                            "answer": answer
                        }
                        print(f"    Answered Q{i}")
                    else:
                        prompt = GENERAL_PROMPT + f"\n\n{question}"
                        answer = generate_inference(prompt, model)
                        results[concept_name][level_name][f"Q{i}"] = {
                            "question": question,
                            "answer": answer
                        }
                        print(f"    Answered Q{i}")
            else:
                # Named questions (Q1_*, Q2_*, etc.) or special types
                for q_key, q_value in level_data.items():
                    if isinstance(q_value, dict):
                        if 'question' in q_value:
                            situation = q_value.get('situation', '')
                            q_text = q_value['question']
                            full_question = f"{situation}\n\n{q_text}" if situation else q_text
                            prompt = GENERAL_PROMPT + f"\n\n{full_question}"
                            answer = generate_inference(prompt, model)
                            results[concept_name][level_name][q_key] = {
                                "question": full_question,
                                "answer": answer
                            }
                            print(f"    Answered {q_key}")
                    else:
                        prompt = GENERAL_PROMPT + f"\n\n{q_value}"
                        answer = generate_inference(prompt, model)
                        results[concept_name][level_name][q_key] = {
                            "question": q_value,
                            "answer": answer
                        }
                        print(f"    Answered {q_key}")
    
    return results

### Step 1: Generate Answers to All Questions

In [ ]:
# Generate answers for all ques}tions in the concept_tasks JSON
results_concept_tasks = answer_all_questions(concept_tasks, model)

# Save the results
os.makedirs('outputs', exist_ok=True)
with open('outputs/results_concepts_tasks.json', 'w', encoding='utf-8') as f:
    json.dump(results_concept_tasks, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("All answers saved to: outputs/results_concepts_tasks.json")
print("="*60)


Processing concept: analogy

  Level: Level_1_Define


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

    Answered Q1
    Answered Q2
    Answered Q3

  Level: Level_2_Application
    Answered Q1_Generate_Example
    Answered Q2_Concrete_Situation

  Level: Level_3_Comprehension
    Answered Q1_Five_Year_Old
    Answered Q2_Teenager
    Answered Q3_University_Student
    Answered Q4_Professor

  Level: Level_4_Analysis_Evaluation
    Answered Q1
    Answered Q2
    Answered Q3

  Level: Level_5_Synthesis_Multihop_Context

  Level: Level_6_Perturbation_Coherence

Processing concept: haiku

  Level: Level_1_Define
    Answered Q1
    Answered Q2
    Answered Q3

  Level: Level_2_Application
    Answered Q1_Generate_Example
    Answered Q2_Concrete_Situation

  Level: Level_3_Comprehension
    Answered Q1_Five_Year_Old
    Answered Q2_Teenager
    Answered Q3_University_Student
    Answered Q4_Professor

  Level: Level_4_Analysis_Evaluation
    Answered Q1
    Answered Q2
    Answered Q3

  Level: Level_5_Synthesis_Multihop_Context

  Level: Level_6_Perturbation_Coherence

Processing conc

### Step 2: Judge the Answers using Gemini

In [7]:
def judge_concept_answers(results_file, judge_model="gemini", delay_seconds=6):
    from google import genai
    from prompts import GENERAL_PROMPT
    import time
    
    # Load the results
    with open(results_file, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    judged_results = {}
    request_count = 0
    
    # Initialize Gemini client
    if judge_model.lower() == "gemini":
        client = genai.Client()
        model_name = "gemini-2.0-flash-exp"
    
    for concept_name, concept_data in results.items():
        print(f"\n{'='*60}")
        print(f"Judging concept: {concept_name}")
        print(f"{'='*60}")
        
        judged_results[concept_name] = {}
        
        for level_name, level_data in concept_data.items():
            print(f"\n  Level: {level_name}")
            judged_results[concept_name][level_name] = {}
            
            for q_key, q_data in level_data.items():
                if isinstance(q_data, dict):
                    if 'question' in q_data and 'answer' in q_data:
                        question = q_data['question']
                        answer = q_data['answer']
                        
                        judge_prompt = f"""You are an impartial judge. Evaluate whether the answer correctly addresses the question about the concept "{concept_name}".
Evaluation Rules:
- Class 1 (CORRECT): The answer accurately addresses the question and demonstrates correct understanding of the concept.
- Class -1 (INCORRECT): The answer is incorrect, contradicts the concept, or fails to properly address the question.

Guidelines:
- Base your judgment on the concept, question, and answer provided.
- Keep reasoning concise (2-3 lines).
- You can think as much as you want but follow the exact answer format below.
Answer Format (no deviations):
FINAL ANSWER: <1 or -1>
Explanation: <2-3 line justification>
Concept: {concept_name}
Question: {question}
Answer: {answer}"""
                        
                        # Add delay to avoid rate limits
                        if request_count > 0:
                            print(f"    Waiting {delay_seconds}s to avoid rate limit...")
                            time.sleep(delay_seconds)
                        
                        # Get judgment
                        response = client.models.generate_content(
                            model=model_name, contents=judge_prompt
                        )
                        judgment = response.text.strip()
                        request_count += 1
                        
                        judged_results[concept_name][level_name][q_key] = {
                            "question": question,
                            "answer": answer,
                            "judgment": judgment
                        }
                        print(f"    Judged {q_key} (Request #{request_count})")
    
    print(f"\n{'='*60}")
    print(f"Total API requests made: {request_count}")
    print(f"{'='*60}")
    
    return judged_results

In [8]:
# Judge all the answers using Gemini
# Note: delay_seconds=6 means 10 requests per minute (within the free tier limit)
# Adjust this value if you have higher quota limits
judged_results = judge_concept_answers('outputs/results_concepts_tasks.json', judge_model="gemini", delay_seconds=6)

# Save the judged results
with open('outputs/judged_results_concepts_tasks.json', 'w', encoding='utf-8') as f:
    json.dump(judged_results, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("All judgments saved to: outputs/judged_results_concepts_tasks.json")
print("="*60)


Judging concept: analogy

  Level: Level_1_Define
    Judged Q1 (Request #1)
    Waiting 6s to avoid rate limit...
    Judged Q2 (Request #2)
    Waiting 6s to avoid rate limit...
    Judged Q3 (Request #3)

  Level: Level_2_Application
    Waiting 6s to avoid rate limit...
    Judged Q1_Generate_Example (Request #4)
    Waiting 6s to avoid rate limit...
    Judged Q2_Concrete_Situation (Request #5)

  Level: Level_3_Comprehension
    Waiting 6s to avoid rate limit...
    Judged Q1_Five_Year_Old (Request #6)
    Waiting 6s to avoid rate limit...
    Judged Q2_Teenager (Request #7)
    Waiting 6s to avoid rate limit...
    Judged Q3_University_Student (Request #8)
    Waiting 6s to avoid rate limit...
    Judged Q4_Professor (Request #9)

  Level: Level_4_Analysis_Evaluation
    Waiting 6s to avoid rate limit...
    Judged Q1 (Request #10)
    Waiting 6s to avoid rate limit...
    Judged Q2 (Request #11)
    Waiting 6s to avoid rate limit...
    Judged Q3 (Request #12)

  Level: Level_

In [9]:
def judge_concept_answers_local(results_file, model):
    from utils import generate_inference
    
    # Load the results
    with open(results_file, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    judged_results = {}
    
    for concept_name, concept_data in results.items():
        print(f"\n{'='*60}")
        print(f"Judging concept: {concept_name}")
        print(f"{'='*60}")
        
        judged_results[concept_name] = {}
        
        for level_name, level_data in concept_data.items():
            print(f"\n  Level: {level_name}")
            judged_results[concept_name][level_name] = {}
            
            for q_key, q_data in level_data.items():
                if isinstance(q_data, dict):
                    if 'question' in q_data and 'answer' in q_data:
                        question = q_data['question']
                        answer = q_data['answer']
                        
                        judge_prompt = f"""You are an impartial judge. Evaluate whether the answer correctly addresses the question about the concept "{concept_name}".

Evaluation Rules:
- Class 1 (CORRECT): The answer accurately addresses the question and demonstrates correct understanding of the concept.
- Class -1 (INCORRECT): The answer is incorrect, contradicts the concept, or fails to properly address the question.

Guidelines:
- Base your judgment on the concept, question, and answer provided.
- Keep reasoning concise (2-3 lines).
- Follow the exact answer format below.

Answer Format (no deviations):
FINAL ANSWER: <1 or -1>
Explanation: <2-3 line justification>

Concept: {concept_name}

Question: {question}

Answer: {answer}"""
                        
                        # Get judgment from local model
                        judgment = generate_inference(judge_prompt, model)
                        
                        judged_results[concept_name][level_name][q_key] = {
                            "question": question,
                            "answer": answer,
                            "judgment": judgment
                        }
                        print(f"    Judged {q_key}")
    
    print(f"\n{'='*60}")
    print("All judgments completed using local model")
    print(f"{'='*60}")
    
    return judged_results

In [10]:
# Judge all the answers using the local model
judged_results_local = judge_concept_answers_local('outputs/results_concepts_tasks.json', model)

# Save the judged results
with open('outputs/judged_results_concepts_tasks_local.json', 'w', encoding='utf-8') as f:
    json.dump(judged_results_local, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("All judgments saved to: outputs/judged_results_concepts_tasks_local.json")
print("="*60)


Judging concept: analogy

  Level: Level_1_Define
    Judged Q1
    Judged Q2
    Judged Q3

  Level: Level_2_Application
    Judged Q1_Generate_Example
    Judged Q2_Concrete_Situation

  Level: Level_3_Comprehension
    Judged Q1_Five_Year_Old
    Judged Q2_Teenager
    Judged Q3_University_Student
    Judged Q4_Professor

  Level: Level_4_Analysis_Evaluation
    Judged Q1
    Judged Q2
    Judged Q3

  Level: Level_5_Synthesis_Multihop_Context

  Level: Level_6_Perturbation_Coherence

Judging concept: haiku

  Level: Level_1_Define
    Judged Q1
    Judged Q2
    Judged Q3

  Level: Level_2_Application
    Judged Q1_Generate_Example
    Judged Q2_Concrete_Situation

  Level: Level_3_Comprehension
    Judged Q1_Five_Year_Old
    Judged Q2_Teenager
    Judged Q3_University_Student
    Judged Q4_Professor

  Level: Level_4_Analysis_Evaluation
    Judged Q1
    Judged Q2
    Judged Q3

  Level: Level_5_Synthesis_Multihop_Context

  Level: Level_6_Perturbation_Coherence

Judging concept

### Step 3: Analyze Results and Calculate Accuracy

In [11]:
def calculate_accuracies(judged_results_file):
    import re
    from collections import Counter
    
    # Load judged results
    with open(judged_results_file, 'r', encoding='utf-8') as f:
        judged_results = json.load(f)
    
    metrics = {}
    overall_stats = {'total': 0, 'correct': 0, 'incorrect': 0}
    
    for concept_name, concept_data in judged_results.items():
        metrics[concept_name] = {
            'levels': {},
            'overall_accuracy': 0,
            'total_questions': 0,
            'correct_answers': 0
        }
        
        concept_total = 0
        concept_correct = 0
        
        for level_name, level_data in concept_data.items():
            level_verdicts = []
            
            for q_key, q_data in level_data.items():
                if isinstance(q_data, dict):
                    if 'judgment' in q_data:
                        # Simple Q&A format
                        judgment_text = q_data['judgment']
                        match = re.search(r'FINAL ANSWER:\s*([-\d]+)', judgment_text)
                        if match:
                            verdict = int(match.group(1))
                            level_verdicts.append(verdict)
            
            # Calculate level metrics
            total = len(level_verdicts)
            correct = sum(1 for v in level_verdicts if v == 1)
            accuracy = correct / total if total > 0 else 0
            
            metrics[concept_name]['levels'][level_name] = {
                'total': total,
                'correct': correct,
                'incorrect': total - correct,
                'accuracy': accuracy
            }
            
            concept_total += total
            concept_correct += correct
        
        # Calculate concept overall metrics
        metrics[concept_name]['total_questions'] = concept_total
        metrics[concept_name]['correct_answers'] = concept_correct
        metrics[concept_name]['overall_accuracy'] = concept_correct / concept_total if concept_total > 0 else 0
        
        overall_stats['total'] += concept_total
        overall_stats['correct'] += concept_correct
        overall_stats['incorrect'] += (concept_total - concept_correct)
    
    # Calculate overall accuracy
    overall_stats['accuracy'] = overall_stats['correct'] / overall_stats['total'] if overall_stats['total'] > 0 else 0
    metrics['_overall_statistics'] = overall_stats
    
    return metrics

In [12]:
# Calculate accuracies and metrics
accuracy_metrics = calculate_accuracies('outputs/judged_results_concepts_tasks_local.json')

# Save the metrics
with open('outputs/accuracy_metrics_concepts_tasks_local.json', 'w', encoding='utf-8') as f:
    json.dump(accuracy_metrics, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("Accuracy metrics saved to: outputs/accuracy_metrics_concepts_tasks_local.json")
print("="*60)

# Display summary
print("\n" + "="*60)
print("OVERALL STATISTICS")
print("="*60)
overall = accuracy_metrics['_overall_statistics']
print(f"Total Questions: {overall['total']}")
print(f"Correct Answers: {overall['correct']}")
print(f"Incorrect Answers: {overall['incorrect']}")
print(f"Overall Accuracy: {overall['accuracy']:.2%}")

print("\n" + "="*60)
print("ACCURACY BY CONCEPT")
print("="*60)
for concept_name, concept_data in accuracy_metrics.items():
    if concept_name != '_overall_statistics':
        print(f"\n{concept_name}:")
        print(f"  Total Questions: {concept_data['total_questions']}")
        print(f"  Correct Answers: {concept_data['correct_answers']}")
        print(f"  Accuracy: {concept_data['overall_accuracy']:.2%}")
        print(f"  Levels:")
        for level_name, level_data in concept_data['levels'].items():
            print(f"    {level_name}: {level_data['accuracy']:.2%} ({level_data['correct']}/{level_data['total']})")


Accuracy metrics saved to: outputs/accuracy_metrics_concepts_tasks_local.json

OVERALL STATISTICS
Total Questions: 120
Correct Answers: 116
Incorrect Answers: 4
Overall Accuracy: 96.67%

ACCURACY BY CONCEPT

analogy:
  Total Questions: 12
  Correct Answers: 12
  Accuracy: 100.00%
  Levels:
    Level_1_Define: 100.00% (3/3)
    Level_2_Application: 100.00% (2/2)
    Level_3_Comprehension: 100.00% (4/4)
    Level_4_Analysis_Evaluation: 100.00% (3/3)
    Level_5_Synthesis_Multihop_Context: 0.00% (0/0)
    Level_6_Perturbation_Coherence: 0.00% (0/0)

haiku:
  Total Questions: 12
  Correct Answers: 11
  Accuracy: 91.67%
  Levels:
    Level_1_Define: 100.00% (3/3)
    Level_2_Application: 100.00% (2/2)
    Level_3_Comprehension: 100.00% (4/4)
    Level_4_Analysis_Evaluation: 66.67% (2/3)
    Level_5_Synthesis_Multihop_Context: 0.00% (0/0)
    Level_6_Perturbation_Coherence: 0.00% (0/0)

paradox:
  Total Questions: 12
  Correct Answers: 11
  Accuracy: 91.67%
  Levels:
    Level_1_Define: 100

In [13]:
# Calculate accuracies and metrics
accuracy_metrics = calculate_accuracies('outputs/judged_results_concepts_tasks.json')

# Save the metrics
with open('outputs/accuracy_metrics_concepts_tasks.json', 'w', encoding='utf-8') as f:
    json.dump(accuracy_metrics, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("Accuracy metrics saved to: outputs/accuracy_metrics_concepts_tasks.json")
print("="*60)

# Display summary
print("\n" + "="*60)
print("OVERALL STATISTICS")
print("="*60)
overall = accuracy_metrics['_overall_statistics']
print(f"Total Questions: {overall['total']}")
print(f"Correct Answers: {overall['correct']}")
print(f"Incorrect Answers: {overall['incorrect']}")
print(f"Overall Accuracy: {overall['accuracy']:.2%}")

print("\n" + "="*60)
print("ACCURACY BY CONCEPT")
print("="*60)
for concept_name, concept_data in accuracy_metrics.items():
    if concept_name != '_overall_statistics':
        print(f"\n{concept_name}:")
        print(f"  Total Questions: {concept_data['total_questions']}")
        print(f"  Correct Answers: {concept_data['correct_answers']}")
        print(f"  Accuracy: {concept_data['overall_accuracy']:.2%}")
        print(f"  Levels:")
        for level_name, level_data in concept_data['levels'].items():
            print(f"    {level_name}: {level_data['accuracy']:.2%} ({level_data['correct']}/{level_data['total']})")


Accuracy metrics saved to: outputs/accuracy_metrics_concepts_tasks.json

OVERALL STATISTICS
Total Questions: 120
Correct Answers: 117
Incorrect Answers: 3
Overall Accuracy: 97.50%

ACCURACY BY CONCEPT

analogy:
  Total Questions: 12
  Correct Answers: 12
  Accuracy: 100.00%
  Levels:
    Level_1_Define: 100.00% (3/3)
    Level_2_Application: 100.00% (2/2)
    Level_3_Comprehension: 100.00% (4/4)
    Level_4_Analysis_Evaluation: 100.00% (3/3)
    Level_5_Synthesis_Multihop_Context: 0.00% (0/0)
    Level_6_Perturbation_Coherence: 0.00% (0/0)

haiku:
  Total Questions: 12
  Correct Answers: 11
  Accuracy: 91.67%
  Levels:
    Level_1_Define: 100.00% (3/3)
    Level_2_Application: 100.00% (2/2)
    Level_3_Comprehension: 100.00% (4/4)
    Level_4_Analysis_Evaluation: 66.67% (2/3)
    Level_5_Synthesis_Multihop_Context: 0.00% (0/0)
    Level_6_Perturbation_Coherence: 0.00% (0/0)

paradox:
  Total Questions: 12
  Correct Answers: 11
  Accuracy: 91.67%
  Levels:
    Level_1_Define: 100.00% (

### Optional: Run Complete Pipeline in One Go

In [ ]:
# Judge all answers
print("\n" + "="*60)
print("STEP 3: Judging all answers...")
print("="*60)
judged_results = judge_concept_answers_local('outputs/results_concepts_tasks.json', model)

# Save judged results
with open('outputs/judged_results_concepts_tasks_local.json', 'w', encoding='utf-8') as f:
    json.dump(judged_results, f, indent=4, ensure_ascii=False)
print("\n✓ Judgments saved to: outputs/judged_results_concepts_tasks_local.json")

# Calculate metrics
print("\n" + "="*60)
print("STEP 4: Calculating accuracy metrics...")
print("="*60)
metrics = calculate_accuracies('outputs/judged_results_concepts_tasks_local.json')

# Save metrics
with open('outputs/accuracy_metrics_concepts_tasks_local.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)
print("\n✓ Metrics saved to: outputs/accuracy_metrics_concepts_tasks_local.json")

---
## Usage Instructions

### Option 1: Run step by step
Execute each cell in order to:
1. Load concept tasks
2. Generate answers
3. Judge with local model
4. Calculate metrics

### Option 2: Run complete pipeline
Execute the cells under "Optional: Run Complete Pipeline in One Go" to run everything at once.